# Reproducible pipeline

The project detects `own`, `leak`, and `third` speech on both channels of an 8 kHz stereo telephone recording and removes leakage. This notebook covers data synthesis, optional training, validation, and inference.

The existing model weights are used by default. Set `TRAIN = True` to retrain the detector and cleaner on the bundled Russian speech data.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

import soundfile as sf

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

TRAIN = False
DIALOGUE_DURATION = 3.0
SEED = 42
ROOT

PosixPath('/home/batoshka/extraneous_speech_detect')

## 1. Data

Training and test dialogues are synthesized from disjoint speaker folders. The synthesizer creates the physical scenarios and ground-truth intervals used by the detector and cleaner.

In [2]:
from src.data_loader import RealSpeechLoader
from src.synthesizer import TelephoneDialogueSynthesizer

loader = RealSpeechLoader()
synthesizer = TelephoneDialogueSynthesizer(seed=SEED)
speakers = loader.discover_speakers()
print('Speakers:', list(speakers))

speaker_a, speaker_b, speaker_third = loader.sample_dialogue_speakers(
    'test', russian_prob=1.0
)
pool = loader.build_utterance_pool(speaker_a, speaker_b, speaker_third, max_per_speaker=4)
example = synthesizer.synthesize_dialogue(
    utterance_pool=pool,
    duration_sec=DIALOGUE_DURATION,
    scenario='normal',
)
print(example.audio.shape, example.sample_rate, example.ground_truth_intervals.keys())

Using Combined Speech Loader: Russian (/home/batoshka/extraneous_speech_detect/data/russian_speech) + VoxCeleb1 (/home/batoshka/extraneous_speech_detect/data/voxceleb1_test)
Speakers: ['ru_spk_1', 'ru_spk_2', 'ru_spk_3', 'ru_spk_4', 'ru_spk_5', 'ru_spk_6', 'ru_spk_7', 'vox_id10270', 'vox_id10271', 'vox_id10272', 'vox_id10273', 'vox_id10274', 'vox_id10275', 'vox_id10276', 'vox_id10277', 'vox_id10278', 'vox_id10279', 'vox_id10280', 'vox_id10281', 'vox_id10282', 'vox_id10283', 'vox_id10284', 'vox_id10285', 'vox_id10286', 'vox_id10287', 'vox_id10288', 'vox_id10289', 'vox_id10290', 'vox_id10291', 'vox_id10292', 'vox_id10293', 'vox_id10294', 'vox_id10295', 'vox_id10296', 'vox_id10297', 'vox_id10298', 'vox_id10299', 'vox_id10300', 'vox_id10301', 'vox_id10302', 'vox_id10303', 'vox_id10304', 'vox_id10305', 'vox_id10306', 'vox_id10307', 'vox_id10308', 'vox_id10309']
(2, 24000) 8000 dict_keys(['A', 'B'])


## 2. Training

The commands call the maintained training entry points and do not alter model architecture or preprocessing. Small values are useful for a smoke test; the defaults in the scripts are intended for a full training run.

In [3]:
def run(command):
    print(' '.join(command))
    subprocess.run(command, cwd=ROOT, check=True)

if TRAIN:
    run([sys.executable, 'scripts/train_detector.py'])
    run([sys.executable, 'scripts/train_cleaner.py'])
else:
    print('Using models/detector_weights.pt and models/cleaner_weights.pt')

Using models/detector_weights.pt and models/cleaner_weights.pt


## 3. Validation

Run the automated tests for the DSP pipeline, model interfaces, cleaner behavior, and CPU real-time constraint. The full scenario evaluation is optional because it synthesizes a larger test set.

In [4]:
run([sys.executable, '-m', 'pytest', 'tests'])

/home/batoshka/extraneous_speech_detect/.venv/bin/python -m pytest tests
============================= test session starts ==============================
platform linux -- Python 3.14.7, pytest-9.1.1, pluggy-1.6.0
rootdir: /home/batoshka/extraneous_speech_detect
configfile: pytest.ini (WARNING: ignoring pytest config in pyproject.toml!)
plugins: platformdirs-4.13.0, anyio-4.15.1
collected 20 items

tests/test_baseline.py .....                                             [ 25%]
tests/test_metrics.py .....                                              [ 50%]
tests/test_stage3.py .....                                               [ 75%]
tests/test_synthesizer.py .....                                          [100%]

============================== 20 passed in 6.17s ==============================


In [5]:
run([sys.executable, 'evaluate.py', '--num_per_scenario', '1', '--duration_sec', '3.0', '--seed', str(SEED)])

/home/batoshka/extraneous_speech_detect/.venv/bin/python evaluate.py --num_per_scenario 1 --duration_sec 3.0 --seed 42
EXTRANEOUS SPEECH DETECTION & CLEANING EVALUATION SUITE
Configuration: 1 dialogues/scenario, 3.0s per dialogue
Hardware: 1 CPU core (OMP_NUM_THREADS=1, torch.set_num_threads(1))
Data: Real Russian speech + VoxCeleb1 (disjoint test speakers) + OpenSLR 28 RIRs
------------------------------------------------------------------------
Using Combined Speech Loader: Russian (data/russian_speech) + VoxCeleb1 (data/voxceleb1_test)
Test speaker pool (3 speakers): ['ru_spk_5', 'ru_spk_6', 'ru_spk_7'] ...
Processed 5/5 dialogues [third_speaker]...]...
EVALUATION RESULTS SUMMARY
Metric                                   | Baseline / Raw   | Our Model / Cleaned  | Delta / Status  
--------------------------------------------------------------------------------------------------
Own F1 (collar 50ms)                     | 0.4541           | 0.5609               | +0.1068
Leak F1 (colla

## 4. Inference

The same saved stereo WAV is passed through both public command-line interfaces. The JSON output contains channel intervals; the WAV output contains the cleaned stereo signal.

In [6]:
input_path = ROOT / 'notebooks' / 'pipeline_input.wav'
json_path = ROOT / 'notebooks' / 'pipeline_predictions.json'
clean_path = ROOT / 'notebooks' / 'pipeline_cleaned.wav'
sf.write(input_path, example.audio.T, example.sample_rate, subtype='PCM_16')
run([sys.executable, 'detect.py', str(input_path), str(json_path)])
run([sys.executable, 'clean.py', str(input_path), str(clean_path)])
print(json.dumps(json.loads(json_path.read_text()), ensure_ascii=False, indent=2)[:1500])
print('Cleaned file:', clean_path)

/home/batoshka/extraneous_speech_detect/.venv/bin/python detect.py /home/batoshka/extraneous_speech_detect/notebooks/pipeline_input.wav /home/batoshka/extraneous_speech_detect/notebooks/pipeline_predictions.json
Predictions successfully written to: /home/batoshka/extraneous_speech_detect/notebooks/pipeline_predictions.json
Detected segments - Track A: 4, Track B: 6
/home/batoshka/extraneous_speech_detect/.venv/bin/python clean.py /home/batoshka/extraneous_speech_detect/notebooks/pipeline_input.wav /home/batoshka/extraneous_speech_detect/notebooks/pipeline_cleaned.wav
Cleaned stereo audio successfully written to: /home/batoshka/extraneous_speech_detect/notebooks/pipeline_cleaned.wav
{
  "A": [
    {
      "start": 0.49,
      "end": 1.38,
      "label": "own",
      "score": 0.9771
    },
    {
      "start": 1.37,
      "end": 1.81,
      "label": "leak",
      "score": 0.8461
    },
    {
      "start": 1.72,
      "end": 1.93,
      "label": "third",
      "score": 0.6544
    },
    